# 04 Honest statistics, and hard negatives

Part 1 computes how often the common intervals cover the true accuracy at the sample sizes this workshop uses (30, 100, 500 items), by exact calculation, not by trusting a formula.
Part 2 puts intervals and the exact McNemar test on the real Meridian test split.
Part 3 adds LLM-written hard negatives to the training set and keeps them only if the held-out result says they helped.

In [1]:
import json
import numpy as np
from pathlib import Path
from scipy.stats import binom, binomtest, norm
from sklearn.linear_model import LogisticRegression
import meridian as m

RESULTS = Path("data/results")
out = {}

## Part 1. Which interval can you trust near the ceiling?
For a true accuracy `p` and `n` test items, the number correct `k` is Binomial(n, p). Every interval is a function of `k` alone, so the exact coverage is the sum of
`P(k)` over the values of `k` whose interval contains `p`. No simulation noise in the Wilson and Wald rows. The percentile bootstrap uses 2,000 resamples per `k` with a fixed seed.

In [2]:
def wald(k, n, z=1.96):
    p = k / n; h = z * np.sqrt(p * (1 - p) / n); return p - h, p + h

def boot(k, n, B=2000, seed=0):
    rng = np.random.default_rng(seed + k)
    s = rng.binomial(n, k / n, size=B) / n
    return np.percentile(s, 2.5), np.percentile(s, 97.5)

def coverage(interval, n, p):
    cov = 0.0
    for k in range(n + 1):
        lo, hi = interval(k, n)
        if lo <= p <= hi: cov += binom.pmf(k, n, p)
    return cov

rows = []
print(f"{'n':>4} {'true acc':>9} {'Wilson':>8} {'Wald':>8} {'bootstrap':>10}")
for n in (30, 100, 500):
    for p in (0.50, 0.80, 0.90, 0.95, 0.98):
        w = coverage(m.wilson, n, p); a = coverage(wald, n, p); b = coverage(boot, n, p)
        rows.append((n, p, w, a, b)); print(f"{n:4d} {p:9.2f} {w:8.3f} {a:8.3f} {b:10.3f}")
wil = [r[2] for r in rows]
print(f"\nWilson coverage over all 15 cells: min {min(wil):.3f}, max {max(wil):.3f}")
print(f"percentile bootstrap at n=30, p=0.98: {[r for r in rows if r[0]==30 and r[1]==0.98][0][4]:.3f};  at n=100, p=0.98: {[r for r in rows if r[0]==100 and r[1]==0.98][0][4]:.3f}")
out["coverage_table"] = rows

   n  true acc   Wilson     Wald  bootstrap
  30      0.50    0.957    0.957      0.957
  30      0.80    0.964    0.946      0.946


  30      0.90    0.974    0.809      0.956
  30      0.95    0.939    0.782      0.782
  30      0.98    0.978    0.454      0.452


 100      0.50    0.943    0.943      0.965


 100      0.80    0.941    0.933      0.963


 100      0.90    0.936    0.932      0.932
 100      0.95    0.966    0.877      0.959


...

 500      0.50    0.946    0.946      0.950


 500      0.80    0.950    0.949      0.961


 500      0.90    0.956    0.943      0.948


 500      0.95    0.950    0.932      0.953


 500      0.98    0.964    0.928      0.965

Wilson coverage over all 15 cells: min 0.936, max 0.978
percentile bootstrap at n=30, p=0.98: 0.452;  at n=100, p=0.98: 0.863


## Part 2. Intervals on the real test split
Meridian's 200 test messages are 40 scenarios of 5 variants. The Wilson interval treats them as 200 independent items; the cluster bootstrap resamples whole scenarios.
The ratio of the two widths is the **design effect** in plain terms: how much smaller the real sample is than it looks.

In [3]:
tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr); _, yte = m.texts_labels(te)
cl = m.cluster_ids(te)
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
preds = {}
for name in ("bge-small", "modernbert", "lfm2.5-enc"):
    E = m.cached_embeddings(name)
    clf = LogisticRegression(C=r01[name]["C"], max_iter=3000).fit(E["train"], ytr)
    preds[name] = clf.predict(E["test"])
for name, p in preds.items():
    ok = (p == yte); k, n = int(ok.sum()), len(ok)
    lo, hi = m.wilson(k, n); clo, chi = m.cluster_bootstrap_ci(ok.astype(float), cl)
    print(f"{name:11s} acc {k/n:.3f}  Wilson width {hi-lo:.3f}  cluster-bootstrap width {chi-clo:.3f}  (ratio {(chi-clo)/(hi-lo):.2f})")

bge-small   acc 0.970  Wilson width 0.050  cluster-bootstrap width 0.065  (ratio 1.30)


modernbert  acc 0.925  Wilson width 0.074  cluster-bootstrap width 0.075  (ratio 1.01)


lfm2.5-enc  acc 0.925  Wilson width 0.074  cluster-bootstrap width 0.110  (ratio 1.49)


### Exact McNemar test: is one system better than the other on the same items?
Only the messages where exactly one system is right carry information. Under "no difference" each discordant message is a fair coin.

In [4]:
def mcnemar(a, b, y):
    ra, rb = (a == y), (b == y)
    n10, n01 = int((ra & ~rb).sum()), int((~ra & rb).sum())
    p = binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
    return n10, n01, p
for a, b in [("bge-small", "modernbert"), ("modernbert", "lfm2.5-enc")]:
    n10, n01, p = mcnemar(preds[a], preds[b], yte)
    print(f"{a} vs {b}: {a} only right {n10}, {b} only right {n01}, exact p = {p:.4f}")
out["mcnemar_encoders"] = {f"{a}|{b}": mcnemar(preds[a], preds[b], yte) for a, b in [("bge-small", "modernbert"), ("modernbert", "lfm2.5-enc")]}
print("Note: the 200 test messages are 40 clusters, so these p-values treat dependent items as independent and are optimistic.")

bge-small vs modernbert: bge-small only right 12, modernbert only right 3, exact p = 0.0352
modernbert vs lfm2.5-enc: modernbert only right 8, lfm2.5-enc only right 8, exact p = 1.0000
Note: the 200 test messages are 40 clusters, so these p-values treat dependent items as independent and are optimistic.


## Part 3. Hard negatives
**Where the baseline fails.** In notebook 01 the main confusion was refund claims routed to payment disputes (the customer says "charged" or "taken from my card" but wants money back for a legitimate purchase).
**The fix tried.** An LLM (Claude Haiku, in a Claude Code session) wrote 80 messages near that line, 40 per route, saved in `data/llm/hard_negatives.jsonl`. They are recorded LLM output, not a live call, and **no person has checked their labels**.
**The risk.** A message the writer meant as a payment dispute may read as a refund claim to a careful person: a hard negative can be an unlabeled positive. The test split stays the original, untouched Meridian split.

In [5]:
hn = [json.loads(l) for l in Path("data/llm/hard_negatives.jsonl").read_text().splitlines()]
print(len(hn), "messages;", {r: sum(h["label"] == r for h in hn) for r in ("refund_claim", "payment_dispute")})
leak = [h for h in hn if "refund_claim" in h["text"] or "payment_dispute" in h["text"]]
print("messages that name their own route:", len(leak))
ENC = "bge-small"
Etr = m.cached_embeddings(ENC)["train"]
Ehn = m.encode(ENC, [h["text"] for h in hn]); yhn = np.array([m.ROUTE_ID[h["label"]] for h in hn])

80 messages; {'refund_claim': 40, 'payment_dispute': 40}
messages that name their own route: 0


/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Filter near-duplicates
Drop any generated message whose cosine similarity to a training message is 0.92 or higher (it adds no information), and any pair within the generated set above that threshold.

In [6]:
THRESH = 0.92
sim_train = (Ehn @ Etr.T).max(1)
keep = np.ones(len(hn), bool); keep[sim_train >= THRESH] = False
S = Ehn @ Ehn.T
for i in range(len(hn)):
    if not keep[i]: continue
    for j in range(i + 1, len(hn)):
        if keep[j] and S[i, j] >= THRESH: keep[j] = False
print(f"kept {keep.sum()} of {len(hn)} (max similarity to training before filtering: median {np.median(sim_train):.2f}, max {sim_train.max():.2f})")

kept 80 of 80 (max similarity to training before filtering: median 0.83, max 0.90)


### Retrain with and without, and compare on the original test split
Same head, same `C`, same embedding model. The only change is the 80-or-fewer extra training messages. The comparison is paired (same test messages), so use the exact McNemar test and a cluster-bootstrap interval on the accuracy difference.

In [7]:
C = r01[ENC]["C"]; Ete = m.cached_embeddings(ENC)["test"]
base = LogisticRegression(C=C, max_iter=3000).fit(Etr, ytr)
aug = LogisticRegression(C=C, max_iter=3000).fit(np.vstack([Etr, Ehn[keep]]), np.concatenate([ytr, yhn[keep]]))
pb, pa = base.predict(Ete), aug.predict(Ete)

def report(name, p):
    ok = (p == yte); k, n = int(ok.sum()), len(ok); lo, hi = m.wilson(k, n); clo, chi = m.cluster_bootstrap_ci(ok.astype(float), cl)
    print(f"{name:22s} acc {k/n:.3f}  Wilson [{lo:.3f}, {hi:.3f}]  cluster [{clo:.3f}, {chi:.3f}]"); return k
report("without hard negatives", pb); report("with hard negatives", pa)
n10, n01, p = mcnemar(pa, pb, yte)
print(f"with-only right {n10}, without-only right {n01}, exact McNemar p = {p:.3f}")

# difference interval, cluster bootstrap over scenarios
d = (pa == yte).astype(float) - (pb == yte).astype(float)
rng = np.random.default_rng(m.SEED); ids = np.unique(cl); per = {c: d[cl == c] for c in ids}
boots = [np.concatenate([per[c] for c in rng.choice(ids, len(ids))]).mean() for _ in range(5000)]
dlo, dhi = np.percentile(boots, [2.5, 97.5])
print(f"accuracy difference (with minus without): {d.mean():+.3f}  cluster-bootstrap 95% [{dlo:+.3f}, {dhi:+.3f}]")
cm = lambda pred: [(yte[i] == m.ROUTE_ID['refund_claim']) & (pred[i] == m.ROUTE_ID['payment_dispute']) for i in range(len(yte))]
print(f"refund claims sent to payment_dispute: without {sum(cm(pb))}, with {sum(cm(pa))} (of {(yte == m.ROUTE_ID['refund_claim']).sum()})")
out["hard_negatives"] = dict(generated=len(hn), kept=int(keep.sum()), acc_without=float((pb == yte).mean()), acc_with=float((pa == yte).mean()),
                             diff=float(d.mean()), diff_cluster_ci=[float(dlo), float(dhi)], mcnemar=[n10, n01, float(p)],
                             refund_to_payment_without=sum(cm(pb)), refund_to_payment_with=sum(cm(pa)))

without hard negatives acc 0.970  Wilson [0.936, 0.986]  cluster [0.930, 0.995]


with hard negatives    acc 0.960  Wilson [0.923, 0.980]  cluster [0.920, 0.990]
with-only right 0, without-only right 2, exact McNemar p = 0.500


accuracy difference (with minus without): -0.010  cluster-bootstrap 95% [-0.030, +0.000]
refund claims sent to payment_dispute: without 3, with 3 (of 25)


## Decision rule
Keep the hard negatives only if the interval on the held-out accuracy difference excludes zero on the favorable side, or if the specific confusion you targeted falls by more than the
interval allows. An interval that straddles zero reads as "no demonstrated effect"; the 80 messages are then dropped, because each one is unchecked data that could carry a wrong label.

In [8]:
(RESULTS / "04_statistics_and_hard_negatives.json").write_text(json.dumps(out, indent=2, default=float))

2184